# 04 - Google Play Store Reviews Ingestion & Data Verification
Tahap Ingestion dan Verifikasi Kualitas Data Mentah Ulasan Google Play Store untuk Aplikasi Ekosistem Zahir (Fase 2).

## 1. Judul dan Tujuan
Notebook ini bertujuan untuk melakukan **ingestion** dan **verifikasi kualitas data awal** terhadap dataset review Google Play Store aplikasi Zahir. Pada tahap ini:
- Tidak dilakukan pembersihan permanen (cleaning/preprocessing final).
- Tidak menghapus baris missing value atau anomali.
- Menjaga dataset mentah tetap asli (*raw data integrity*).
- Memetakan struktur kolom, missing values, duplikasi, distribusi rating, rentang waktu, dan respon pihak perusahaan sebagai dasar ilmiah sebelum preprocessing Fase 3.

## 2. Data Source
- **Platform**: Google Play Store
- **Target Aplikasi**: Ekosistem Aplikasi Zahir (Zahir Simple Start, Zahir Online, POS X, Zahir Certification, Zahir Attendance)
- **Lokasi File Sumber**: `D:\FILE KULIAH\semester 5\magang\Zahir_PlayStore\data\raw\zahir_playstore_raw.csv`
- **Target Relatif Proyek**: `data/raw/playstore_reviews_raw.csv`

## 3. Import Library
Mengimpor library manipulasi data dan mengatur opsi tampilan.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print('Python Executable:', sys.executable)
print('Pandas Version:', pd.__version__)

Python Executable: D:\FILE KULIAH\semester 5\magang\zahir_social_analytics\.venv_social\Scripts\python.exe
Pandas Version: 3.0.6


## 4. Load Raw Dataset
Membaca dataset Play Store dari path target atau path sumber asli yang teridentifikasi secara aman.

In [2]:
target_path = '../data/raw/playstore_reviews_raw.csv'
discovered_path = r'D:\FILE KULIAH\semester 5\magang\Zahir_PlayStore\data\raw\zahir_playstore_raw.csv'

if os.path.exists(target_path):
    dataset_path = target_path
    print(f'Memuat dataset dari target path: {dataset_path}')
elif os.path.exists(discovered_path):
    dataset_path = discovered_path
    print(f'Memuat dataset dari discovered path: {dataset_path}')
else:
    raise FileNotFoundError('Dataset Play Store tidak ditemukan!')

df = pd.read_csv(dataset_path)
print(f'Dataset berhasil dimuat dengan shape: {df.shape}')

Memuat dataset dari discovered path: D:\FILE KULIAH\semester 5\magang\Zahir_PlayStore\data\raw\zahir_playstore_raw.csv
Dataset berhasil dimuat dengan shape: (417, 13)


## 5. Dataset Overview
Menampilkan ukuran data, tipe data awal, serta 5 baris pertama (dengan kolom anonim untuk privasi).

In [3]:
print(f'Jumlah Baris : {df.shape[0]}')
print(f'Jumlah Kolom : {df.shape[1]}')
print('\nInfo Dataset:')
df.info()

print('\nContoh Data Awal (Head):')
# Tampilkan head tanpa mengekspos PII userName secara penuh jika diperlukan
display(df[['reviewId', 'score', 'thumbsUpCount', 'at', 'app_name', 'package_name']].head())

Jumlah Baris : 417
Jumlah Kolom : 13

Info Dataset:
<class 'pandas.DataFrame'>
RangeIndex: 417 entries, 0 to 416
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   reviewId              417 non-null    str  
 1   userName              417 non-null    str  
 2   userImage             417 non-null    str  
 3   content               417 non-null    str  
 4   score                 417 non-null    int64
 5   thumbsUpCount         417 non-null    int64
 6   reviewCreatedVersion  312 non-null    str  
 7   at                    417 non-null    str  
 8   replyContent          366 non-null    str  
 9   repliedAt             366 non-null    str  
 10  appVersion            312 non-null    str  
 11  app_name              417 non-null    str  
 12  package_name          417 non-null    str  
dtypes: int64(2), str(11)
memory usage: 42.5 KB

Contoh Data Awal (Head):


,reviewId,score,thumbsUpCount,at,app_name,package_name
0,db013820-004c-4ad3-82fd-4fb18de20967,5,0,2025-10-08 10:33:00,Zahir Attendance,com.zahir.attendance
1,8817939a-5285-4ebf-99cb-1a04b1ac2862,1,0,2025-10-06 00:59:53,Zahir Attendance,com.zahir.attendance
2,d21ab8dd-1042-4fca-8985-106d702cdb60,3,0,2025-09-29 07:53:03,Zahir Attendance,com.zahir.attendance
3,1d4508ff-30a4-4a18-b6f3-2aae38f5b9a7,1,0,2025-08-28 19:16:10,Zahir Attendance,com.zahir.attendance
4,ddc0eb85-82f8-46da-a285-e735a90ae1b3,5,0,2025-07-11 10:22:24,Zahir Attendance,com.zahir.attendance


## 6. Schema Validation
Membandingkan kolom aktual dengan daftar kolom yang diharapkan (*expected schema*).

In [4]:
expected_cols = [
    'reviewId', 'userName', 'userImage', 'content', 'score',
    'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent',
    'repliedAt', 'appVersion', 'app_name', 'package_name'
]
actual_cols = df.columns.tolist()

missing_expected = [c for c in expected_cols if c not in actual_cols]
additional_cols = [c for c in actual_cols if c not in expected_cols]

print('Expected columns count :', len(expected_cols))
print('Actual columns count   :', len(actual_cols))
print('Missing expected cols  :', missing_expected)
print('Additional cols        :', additional_cols)
print('Apakah skema 100% cocok:', len(missing_expected) == 0 and len(additional_cols) == 0)

Expected columns count : 13
Actual columns count   : 13
Missing expected cols  : []
Additional cols        : []
Apakah skema 100% cocok: True


## 7. Missing Value Analysis
Menghitung jumlah dan persentase nilai kosong (*null/NaN*) pada setiap kolom tanpa melakukan imputasi atau penghapusan baris.

In [5]:
missing_df = pd.DataFrame({
    'column': df.columns,
    'missing_count': df.isna().sum().values,
    'missing_percentage': (df.isna().mean() * 100).round(2).values
}).sort_values('missing_count', ascending=False)

print('Tabel Ringkasan Missing Values:')
display(missing_df)

Tabel Ringkasan Missing Values:


,column,missing_count,missing_percentage
6,reviewCreatedVersion,105,25.18
10,appVersion,105,25.18
9,repliedAt,51,12.23
8,replyContent,51,12.23
0,reviewId,0,0.00
4,score,0,0.00
3,content,0,0.00
2,userImage,0,0.00
1,userName,0,0.00
7,at,0,0.00


## 8. Duplicate Analysis
Memeriksa apakah terdapat duplikasi data pada keseluruhan baris maupun pada kunci unik `reviewId`.

In [6]:
total_dup_rows = df.duplicated().sum()
dup_review_id = df['reviewId'].duplicated().sum()

print(f'Total duplikasi baris penuh   : {total_dup_rows}')
print(f'Total duplikasi reviewId      : {dup_review_id}')

Total duplikasi baris penuh   : 0
Total duplikasi reviewId      : 0


## 9. Score (Rating) Validation
Memverifikasi distribusi skor rating (1 sampai 5) dan memastikan tidak ada anomali atau skor di luar rentang standar.

In [7]:
score_dist = df['score'].value_counts(dropna=False).sort_index()
score_out_of_range = ((df['score'] < 1) | (df['score'] > 5)).any()

print('Distribusi Rating Score:')
print(score_dist)
print(f'\nTipe data score            : {df["score"].dtype}')
print(f'Apakah ada rating di luar 1-5 : {score_out_of_range}')

Distribusi Rating Score:
score
1     74
2     27
3     37
4     31
5    248
Name: count, dtype: int64

Tipe data score            : int64
Apakah ada rating di luar 1-5 : False


## 10. Thumbs Up Validation
Memeriksa metrik `thumbsUpCount` untuk memastikan tidak ada nilai negatif atau anomali statistik.

In [8]:
thumbs_stats = df['thumbsUpCount'].describe()
has_negative_thumbs = (df['thumbsUpCount'] < 0).any()

print('Statistik Deskriptif thumbsUpCount:')
print(thumbs_stats)
print(f'Apakah ada nilai negatif : {has_negative_thumbs}')

Statistik Deskriptif thumbsUpCount:
count    417.000000
mean       1.717026
std        3.846392
min        0.000000
25%        0.000000
50%        0.000000
75%        2.000000
max       40.000000
Name: thumbsUpCount, dtype: float64
Apakah ada nilai negatif : False


## 11. Date Validation
Memvalidasi konversi waktu review (`at`) dan respon developer (`repliedAt`) tanpa menimpa kolom mentah.

In [9]:
review_dates = pd.to_datetime(df['at'], errors='coerce')
reply_dates = pd.to_datetime(df['repliedAt'], errors='coerce')

print('=== Validasi Tanggal Review (at) ===')
print(f'Tanggal valid    : {review_dates.notna().sum()}')
print(f'Tanggal invalid  : {review_dates.isna().sum()}')
print(f'Tanggal paling awal: {review_dates.min()}')
print(f'Tanggal paling akhir: {review_dates.max()}')

print('\n=== Validasi Tanggal Balasan (repliedAt) ===')
print(f'Tanggal valid    : {reply_dates.notna().sum()}')
print(f'Missing reply    : {reply_dates.isna().sum()}')
print(f'Tanggal paling awal: {reply_dates.min()}')
print(f'Tanggal paling akhir: {reply_dates.max()}')

=== Validasi Tanggal Review (at) ===
Tanggal valid    : 417
Tanggal invalid  : 0
Tanggal paling awal: 2018-01-26 23:03:08
Tanggal paling akhir: 2026-09-11 14:24:43

=== Validasi Tanggal Balasan (repliedAt) ===
Tanggal valid    : 366
Missing reply    : 51
Tanggal paling awal: 2018-01-27 09:41:34
Tanggal paling akhir: 2026-09-11 14:39:16


## 12. Text Validation
Memeriksa panjang karakter ulasan pengguna (`content`) secara sementara (*in-memory*) tanpa menambahkan kolom permanen ke raw dataset.

In [10]:
text_length = df['content'].fillna('').astype(str).str.len()
empty_text_count = (text_length == 0).sum()

print('Statistik Panjang Teks Review (Karakter):')
print(f'Minimum : {text_length.min()}')
print(f'Median  : {text_length.median()}')
print(f'Mean    : {round(text_length.mean(), 2)}')
print(f'Maximum : {text_length.max()}')
print(f'Jumlah teks kosong (0 chars): {empty_text_count}')

Statistik Panjang Teks Review (Karakter):
Minimum : 2
Median  : 44.0
Mean    : 73.6
Maximum : 612
Jumlah teks kosong (0 chars): 0


## 13. App & Package Validation
Memverifikasi bahwa dataset benar-benar berasal dari aplikasi resmi Zahir.

In [11]:
print('Distribusi Nama Aplikasi (app_name):')
print(df['app_name'].value_counts(dropna=False))

print('\nDistribusi Package ID (package_name):')
print(df['package_name'].value_counts(dropna=False))

Distribusi Nama Aplikasi (app_name):
app_name
Zahir Simple Start     160
Zahir Online            98
POS X                   87
Zahir Certification     44
Zahir Attendance        28
Name: count, dtype: int64

Distribusi Package ID (package_name):
package_name
com.zahirsimplestart      160
com.zahironline            98
com.posx                   87
com.zahircertification     44
com.zahir.attendance       28
Name: count, dtype: int64


## 14. App Version Validation
Memeriksa sebaran versi aplikasi saat ulasan dibuat (`reviewCreatedVersion`) dan versi aplikasi terdaftar (`appVersion`).

In [12]:
print(f'Jumlah versi unik (appVersion): {df["appVersion"].nunique(dropna=False)}')
print('Top 10 appVersion:')
print(df['appVersion'].value_counts(dropna=False).head(10))

print(f'\nJumlah versi unik (reviewCreatedVersion): {df["reviewCreatedVersion"].nunique(dropna=False)}')
print('Top 10 reviewCreatedVersion:')
print(df['reviewCreatedVersion'].value_counts(dropna=False).head(10))

Jumlah versi unik (appVersion): 73
Top 10 appVersion:
appVersion
NaN      105
1.0      104
1.3       30
1.2.5     15
1.5.3      9
1.2.3      8
1.2.2      8
1.4.5      7
1.4.2      6
1.2.0      6
Name: count, dtype: int64

Jumlah versi unik (reviewCreatedVersion): 73
Top 10 reviewCreatedVersion:
reviewCreatedVersion
NaN      105
1.0      104
1.3       30
1.2.5     15
1.5.3      9
1.2.3      8
1.2.2      8
1.4.5      7
1.4.2      6
1.2.0      6
Name: count, dtype: int64


## 15. Company Reply Validation
Menganalisis tingkat respon pengembang (*developer response rate*) terhadap ulasan pengguna.

In [13]:
has_reply = df['replyContent'].notna()
reply_count = has_reply.sum()
no_reply_count = (~has_reply).sum()
reply_pct = round(has_reply.mean() * 100, 2)

print(f'Jumlah ulasan dengan balasan developer : {reply_count}')
print(f'Jumlah ulasan tanpa balasan developer  : {no_reply_count}')
print(f'Persentase ulasan yang mendapat balasan: {reply_pct}%')

Jumlah ulasan dengan balasan developer : 366
Jumlah ulasan tanpa balasan developer  : 51
Persentase ulasan yang mendapat balasan: 87.77%


## 16. Data Quality Summary
Ringkasan temuan kualitas data:
- **Integritas Baris & ID**: 417 baris tanpa ada duplikasi `reviewId` atau duplikasi baris penuh.
- **Kelengkapan Kolom**: Skema 13 kolom 100% cocok dengan skema yang diharapkan.
- **Rating**: 100% bernilai integer 1-5 dengan rating 5 mendominasi (59.47%), diikuti rating 1 (17.75%).
- **Respon Perusahaan**: Responsivitas pengembang sangat tinggi (87.77% review dibalas).
- **Missing Values**: Missing values terpusat pada `reviewCreatedVersion`/`appVersion` (25.18%) dan `replyContent`/`repliedAt` (12.23%), yang wajar secara fungsional di Play Store.

## 17. Kesimpulan Fase 2
Tahap Ingestion dan Verifikasi Fase 2 telah berhasil diselesaikan:
1. Dataset terbukti valid dan autentik dari 5 aplikasi resmi ekosistem Zahir.
2. Dataset mentah tetap utuh tanpa modifikasi/imputasi permanen.
3. Siap untuk dilanjutkan ke tahap pembersihan teks dan preprocessing terstruktur pada Fase 3.